# Demo 05 - Duplicates and Cardinality

A few times now, we've seen interesting behavior with respect to increases in expenses from the years 2019-2021.  Now let's use a couple of tools at our disposal to delve further into the problem.

In [ ]:
import pyodbc
import pandas as pd
import toml

config = toml.load("../config.toml")
server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+password)

## Count of Invoices by Day

The first thing we want to do is see how many invoices we get per bus, vendor, and day.  Our organization requires vendors send one invoice per bus maintenance item, so we generally expect no more than one invoice per combination of bus, vendor, and day.  After all, invoices can get lost in the shuffle and creating more than one is inefficient.

In [ ]:
dupes = pd.read_sql("""WITH records AS
(
	SELECT
		li.LineItemDate,
		li.BusID,
		li.VendorID,
        v.VendorName,
		COUNT(*) AS NumberOfInvoices
	FROM dbo.LineItem li
        INNER JOIN dbo.Vendor v
            ON li.VendorID = v.VendorID
	GROUP BY
		li.LineItemDate,
		li.BusID,
		li.VendorID,
        v.VendorName
)
SELECT
	NumberOfInvoices,
	COUNT(*) AS NumberOfOccurrences
FROM records
GROUP BY
	NumberOfInvoices
ORDER BY
	NumberOfInvoices;""", conn)

In [ ]:
dupes

It looks like, for the vast majority of the time, we see one invoice per combination of bus, vendor, and day. 1220 times we have two invoices, 4 times we have three invoices on a single day, and we got four invoices once. It might be interesting to see who's sending us multiple invoices, so let's do that.

In [ ]:
dupe_senders = pd.read_sql("""WITH records AS
(
	SELECT
		li.LineItemDate,
		li.BusID,
		li.VendorID,
        v.VendorName,
		COUNT(*) AS NumberOfInvoices
	FROM dbo.LineItem li
        INNER JOIN dbo.Vendor v
            ON li.VendorID = v.VendorID
	GROUP BY
		li.LineItemDate,
		li.BusID,
		li.VendorID,
        v.VendorName
)
SELECT
	VendorID,
    VendorName,
	COUNT(*) AS NumberOfOccurrences
FROM records
WHERE
    NumberOfInvoices > 1
GROUP BY
	VendorID,
    VendorName
ORDER BY
	VendorID ASC;""", conn)

In [ ]:
dupe_senders

Vendors 5 and 9 have the most duplicated invoice days, with vendors 8 and 12 quite a ways behind.  This pattern isn't especially weird, so let's keep digging.

## Cardinality Checks

The Pandas library has functionality to help us perform cardinality checks, including showing us how many distinct values there are in our data set.

In [ ]:
raw_data = pd.read_sql("""SELECT
    c.CalendarYear,
    c.CalendarMonth,
    b.BusID,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    ec.ExpenseCategory,
    v.VendorName,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    li.Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID;""", conn)

In [ ]:
raw_data.apply(pd.Series.nunique)

# You can also do this as a foreach loop:
#for col in raw_data:
#    cardinality = len(pd.Index(raw_data[col]).value_counts())
#    print(raw_data[col].name + ": " + str(cardinality))

Looking at the results, we knew that there would be 15 vendors, 28 expense categories, 5 bus models (with their own model qualities), 12 employees, 12 months, and 12 years, so those aren't surprising. We did learn that 1507 buses had line items associated with them. This means that 53 buses have never had maintenance done on them.

Cardinality is also useful when looking at subsets of data.  For example, let's filter to include just invoices valued between 850 USD and 999.99 USD, as these are high-value invoices which fall below the two-signer rule.

In [ ]:
high_value_invoices = raw_data.query('Amount >= 850 & Amount < 1000')
high_value_invoices.apply(pd.Series.nunique)

Based off of this, it looks like 12 of our 15 vendors had invoices between 850 and 999.99.  We can dig a bit deeper and look at counts by vendor.

In [ ]:
high_value_invoices[["VendorName", "Amount"]].groupby("VendorName").count()

There's a pretty nice mixture here. But let's look a bit closer to the edge and say 995 USD or more, but less than 1000 USD.

In [ ]:
high_value_invoices = raw_data.query('Amount >= 995 & Amount < 1000')
high_value_invoices[["VendorName", "Amount"]].groupby("VendorName").count()

Glass and Sons is a bit high here. Let's take a quick look at their invoices in this range:

In [ ]:
high_value_invoices[["VendorName", "ExpenseCategory", "Amount"]].query("VendorName.str.startswith('Glass and Sons')")

A common pattern here is 999.99 USD. Let's see who else shows up with that amount or within a few cents:

In [ ]:
high_value_invoices = raw_data.query('Amount >= 999 & Amount < 1000')
high_value_invoices[["VendorName", "Amount"]].groupby("VendorName").count()

That's perhaps a little weird, but let's see how they look when it comes to invoices of 1000 USD or more. What's important is that these invoices all require two signers.

In [ ]:
high_value_invoices = raw_data.query('Amount >= 1000')
high_value_invoices[["VendorName", "Amount"]].groupby("VendorName").count()

Glass and Sons has only a few invoices over 1000 USD and almost all of the 999(ish) USD invoices.

By the way, let's take a quick look at their invoices by amount for Glass and Sons.

In [ ]:
raw_data[["VendorName","Amount"]] \
    .query('VendorName.str.startswith("Glass and Sons") & Amount >= 950 & Amount < 1000') \
    .groupby("Amount").count()

Err...maybe they offer discounts?  This is starting to look weird.

Now let's do the same sort of analysis by employee.

In [ ]:
high_value_invoices = raw_data.query('Amount >= 900 & Amount < 1000')
high_value_invoices[["EmployeeName", "Amount"]].groupby("EmployeeName").count()

All twelve of our employees have dealt with high-value invoices.  Let's key it down to Glass and Sons.

In [ ]:
high_value_invoices = raw_data.query('VendorName.str.startswith("Glass and Sons") & Amount >= 900 & Amount < 1000')
high_value_invoices[["EmployeeName", "Amount"]].groupby("EmployeeName").count()

Only four employees handled those invoices for vendor 5.  But maybe the agency has people focus on certain sets of vendors.  Let's limit ourselves to the years 2019-2021 and see how many invoices for vendor 5 each employee has handled:

In [ ]:
high_value_invoices = raw_data.query('VendorName.str.startswith("Glass and Sons") & CalendarYear.isin([2019, 2020, 2021])')
high_value_invoices[["EmployeeName", "Amount"]].groupby("EmployeeName").count()

Looks like Villiers, Sweeting, Harte, and Mowett are quite in-demand for working with Glass and Sons, but all 12 have handled invoices. As a quick note, over a three-year period, 8 of the 12 people average approximately 110-120 invoices per year, or once every 3 days. Villiers, Sweeting, Harte, and Mowett handle just under **1500** per year, or an average of 4-5 **per day**.  That's weird.

But maybe it's "normal" weird--so let's compare 2011-2018 and 2022.

In [ ]:
high_value_invoices = raw_data.query('VendorName.str.startswith("Glass and Sons") & CalendarYear.isin([2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2022])')
high_value_invoices[["EmployeeName", "Amount"]].groupby("EmployeeName").count()

So, yeah, it's weird. There was a huge burst in 2019-2021 for those four employees--and only those employees--and the rest of the years, it looks like invoices are fairly evenly spread across all of the employees.